In [ ]:
# Cell 1: imports and project-root setup
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'src').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.riql import RIQLAgent
from src.train_eval import train_riql_from_loader, eval_policy_on_env

In [ ]:
# Cell 2: experiment configuration

METHOD = 'raw_noisy_riql'

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace('.', 'p')
    return f'nd{noise_dim}_ns{ns}_{noise_type}'

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f'seed_{SEED}'

CKPT_DIR    = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_STATS_PATH = OBS_STATS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG / 'obs_stats.npz'

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEVICE:', DEVICE)
print('METHOD:', METHOD)
print('CKPT_DIR:', CKPT_DIR)
print('METRICS_DIR:', METRICS_DIR)

In [ ]:
# Cell 3: dataset and dataloader

dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

state_dim      = dataset.noisy_obs.shape[1]   # corrupted obs dimension
action_dim     = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim              # clean obs dimension

print('state_dim (noisy):', state_dim)
print('true_state_dim:', true_state_dim)
print('action_dim:', action_dim)

if not OBS_STATS_PATH.exists():
    np.savez(
        OBS_STATS_PATH,
        obs_mean=dataset.obs_mean,
        obs_std=dataset.obs_std,
        true_state_dim=true_state_dim,
    )
    print('Saved:', OBS_STATS_PATH)
else:
    print('Already exists:', OBS_STATS_PATH)

In [ ]:
# Cell 4: create and train RIQLAgent
#
# RIQL differences from IQL:
#   - n_critics=10 ensemble instead of twin Q
#   - Huber loss for critic (robust to heavy-tailed corrupted data)
#   - quantile=0.25 pessimistic value target over ensemble

riql = RIQLAgent(
    latent_dim=state_dim,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
    tau=0.005,
    lr=3e-4,
    n_critics=10,
    quantile=0.25,
    huber_delta=1.0,
)

history = train_riql_from_loader(
    riql=riql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    use_tqdm=True,
)

In [ ]:
# Cell 5: evaluate and save metrics

metrics = eval_policy_on_env(
    iql=riql,
    env_name=ENV_NAME,
    encoder=None,
    method='raw_noisy',   # obs handling: encoder=None -> z = noisy obs directly
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = METRICS_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'env': ENV_NAME,
            'method': METHOD,
            'seed': SEED,
            'noise_dim': NOISE_DIM,
            'noise_scale': NOISE_SCALE,
            'noise_type': NOISE_TYPE,
            **metrics,
        },
        f,
        indent=2,
    )

print('Saved metrics:', metrics_path)
print(metrics)